# 01 — Train CA-HGTN and Ablation Variants

All results are written to a single `{CITY}_predictions.pkl` for
`02_Evaluate_and_Ablate.ipynb`.

In [ ]:
import sys, os

candidates = [os.path.abspath("."), os.path.abspath("..")]
found = next((c for c in candidates
              if os.path.isfile(os.path.join(c, "cahgtn", "__init__.py"))), None)
if found is None:
    raise ModuleNotFoundError(
        f"Could not find a 'cahgtn' package folder in {candidates[0]} or "
        f"{candidates[1]}. See 00_Data_Pipeline.ipynb for the expected layout."
    )
sys.path.insert(0, found)
print(f"Using cahgtn package from: {os.path.join(found, 'cahgtn')}")

import numpy as np
import torch
from torch.utils.data import DataLoader

from cahgtn.model import CAHGTN, GRUOnlyBaseline
from cahgtn.baselines import GCNGRUBaseline
from cahgtn.train import (
    train_one_run, train_fixed_epochs, predict_all, WindowDataset,
    search_patience_and_ceiling,
)

# ================================================================
# City Selection
# ================================================================
CITY = "Beijing"        # "Hangzhou" | "Shanghai" | "Beijing"
N_SEEDS = 5 
EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 32

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

d = np.load(f"{CITY}_windows.npz")
adj_physical = torch.tensor(d["adjacency_physical"])
adj_corr = torch.tensor(d["adjacency_corr"])

HAVE_PRECOMPUTED_GRAPHS = "adjacency_similarity_precomputed" in d.files
if HAVE_PRECOMPUTED_GRAPHS:
    adj_similarity_precomputed = torch.tensor(d["adjacency_similarity_precomputed"])
    adj_correlation_precomputed = torch.tensor(d["adjacency_correlation_precomputed"])
else:
    print(f"NOTE: {CITY}_windows.npz has no precomputed similarity/correlation graphs "
          f"-- re-run notebook 00 for {CITY} to add them. Only Section B's "
          f"precomputed-graph ablation needs them; everything else is unaffected.")
    adj_similarity_precomputed = adj_correlation_precomputed = None

num_stations = int(d["num_stations"])
horizon = d["Y_train"].shape[1]
print("num_stations:", num_stations, "horizon:", horizon)

def make_loader_tuple(prefix):
    return (d[f"X_{prefix}"], d[f"Y_{prefix}"], d[f"TF_{prefix}"], d[f"EX_{prefix}"], d[f"SEAS_{prefix}"])

internal_fit = make_loader_tuple("ifit")
internal_val = make_loader_tuple("ival")
official_train = make_loader_tuple("train")
official_test = make_loader_tuple("test")
naive_test = d["naive_test"]

bias_init_official = (float(official_train[1][..., 0].mean()), float(official_train[1][..., 1].mean()))
print(f"official_train mean (inflow, outflow) per 15min: {bias_init_official}")

all_predictions = {}
chosen_epochs = {}
head_type_for = {}

## Parameter counts (Table V)

Computed directly, no training needed.

In [ ]:
%%time

from cahgtn.param_counter import cahgtn_params, gru_only_baseline_params, gcn_gru_baseline_params

param_counts = {
    "CA-HGTN (full)": cahgtn_params(hidden_dim=64, num_stations=num_stations, horizon=horizon),
    "GRU-only baseline": gru_only_baseline_params(hidden_dim=64, horizon=horizon),
    "GCN+GRU baseline": gcn_gru_baseline_params(hidden_dim=64, horizon=horizon),
}
for name, n in param_counts.items():
    print(f"{name}: {n:,} parameters")

## Phase 1 — learning-rate selection

In [ ]:
%%time

LR_CANDIDATES = [1e-3, 3e-4]

bias_init_ifit = (float(internal_fit[1][..., 0].mean()), float(internal_fit[1][..., 1].mean()))

def quick_val_loss(model_ctor, head_type, lr):
    m, _ = train_one_run(model_ctor, internal_fit, internal_val, adj_physical, adj_corr,
                          device, head_type=head_type, epochs=25, batch_size=BATCH_SIZE,
                          lr=lr, patience=6, seed=0)
    val_loader = DataLoader(WindowDataset(*internal_val), batch_size=BATCH_SIZE, shuffle=False)
    preds, trues = predict_all(m, val_loader, adj_physical.to(device), adj_corr.to(device), device, head_type)
    key = "mu" if head_type == "nb" else "pred"
    return np.mean(np.abs(preds[key] - trues))

variants = {
    "CA-HGTN (full)":         dict(head_type="nb",    graph_mode="full",          use_seasonal_residual=True),
    "CA-HGTN (point head)":   dict(head_type="point", graph_mode="full",          use_seasonal_residual=True),
    "CA-HGTN (single-graph)": dict(head_type="nb",    graph_mode="physical_only", use_seasonal_residual=True),
    "CA-HGTN (no-physical)":  dict(head_type="nb",    graph_mode="no_physical",   use_seasonal_residual=True),
}

def ctor_for_lr_search(cfg):
    return lambda: CAHGTN(num_stations=num_stations, hidden_dim=64, horizon=horizon,
                           head_type=cfg["head_type"], graph_mode=cfg["graph_mode"],
                           use_seasonal_residual=cfg["use_seasonal_residual"],
                           output_bias_init=bias_init_ifit)

best_lr = {}
for name, cfg in variants.items():
    scores = {lr: quick_val_loss(ctor_for_lr_search(cfg), cfg["head_type"], lr) for lr in LR_CANDIDATES}
    best_lr[name] = min(scores, key=scores.get)
    print(f"{name}: internal-val MAE per lr = {scores}  -> chosen lr={best_lr[name]}")
best_lr["GRU-only baseline"] = 1e-3
best_lr["GCN+GRU baseline"] = 1e-3

ALL_VARIANT_NAMES = list(variants) + ["GRU-only baseline", "GCN+GRU baseline"]

def ctor_for(name):
    if name == "GRU-only baseline":
        return lambda: GRUOnlyBaseline(hidden_dim=64, horizon=horizon,
                                        output_bias_init=bias_init_official, use_seasonal_residual=True)
    if name == "GCN+GRU baseline":
        return lambda: GCNGRUBaseline(num_stations=num_stations, hidden_dim=64, horizon=horizon,
                                       output_bias_init=bias_init_official, use_seasonal_residual=True)
    cfg = variants[name]
    return lambda: CAHGTN(num_stations=num_stations, hidden_dim=64, horizon=horizon,
                           head_type=cfg["head_type"], graph_mode=cfg["graph_mode"],
                           use_seasonal_residual=cfg["use_seasonal_residual"],
                           output_bias_init=bias_init_official)

for name in ALL_VARIANT_NAMES:
    head_type_for[name] = variants[name]["head_type"] if name in variants else "point"
print("Phase 1 complete.")

## Section A — Main Results (Table III): standard protocol

In [ ]:
%%time

import time

epoch_times = {}

def run_standard_protocol(name, epochs=EPOCHS, patience=PATIENCE):
    print(f"\n=== Training {name} (standard protocol) ===")
    all_predictions[name] = []
    chosen_epochs[name] = []
    epoch_times[name] = []
    for seed in range(N_SEEDS):
        _, best_epoch = train_one_run(
            ctor_for(name), internal_fit, internal_val, adj_physical, adj_corr, device,
            head_type=head_type_for[name], epochs=epochs, batch_size=BATCH_SIZE,
            lr=best_lr[name], patience=patience, seed=seed,
        )
        chosen_epochs[name].append(best_epoch)

        t0 = time.time()
        model = train_fixed_epochs(
            ctor_for(name), official_train, adj_physical, adj_corr, device,
            head_type=head_type_for[name], epochs=max(best_epoch, 1), batch_size=BATCH_SIZE,
            lr=best_lr[name], seed=seed,
        )
        elapsed = time.time() - t0
        epoch_times[name].append(elapsed / max(best_epoch, 1))

        test_loader = DataLoader(WindowDataset(*official_test), batch_size=BATCH_SIZE, shuffle=False)
        preds, trues = predict_all(model, test_loader, adj_physical.to(device), adj_corr.to(device),
                                    device, head_type_for[name])
        all_predictions[name].append((preds, trues))
        torch.save(model.state_dict(),
                   f"{CITY}_{name.replace(chr(32),chr(95)).replace(chr(40),chr(39)).replace(chr(41),chr(39))}_seed{seed}.pt")
    print(f"  epoch budgets across seeds: {chosen_epochs[name]}")
    times_str = ", ".join("%.2f" % t for t in epoch_times[name])
    print(f"  seconds/epoch across seeds: {times_str}")

for name in ALL_VARIANT_NAMES:
    run_standard_protocol(name)
print("\nSection A (standard protocol) complete.")

In [ ]:
%%time

import io, contextlib, pickle, time as _time

CANDIDATES = [
    (100, 175, 350),
    (250, 300, 500),
    (350, 350, 600),
]

EXTENDED_TRAINING_VARIANTS = ["CA-HGTN (full)", "GCN+GRU baseline"]

EXT_N_SEEDS = 3 

checkpoint_path = f"{CITY}_extended_checkpoint.pkl"
try:
    with open(checkpoint_path, "rb") as f:
        checkpoint = pickle.load(f)
    done_search = list(checkpoint["extended_config"].keys())
    done_refit = list(checkpoint["all_predictions_ext"].keys())
    print(f"Resuming from {checkpoint_path}: {done_search} already searched, {done_refit} already refit.")
except FileNotFoundError:
    checkpoint = {"extended_config": {}, "all_predictions_ext": {}, "chosen_epochs_ext": {}}

extended_config = checkpoint["extended_config"]
all_predictions_ext = checkpoint["all_predictions_ext"]
chosen_epochs_ext = checkpoint["chosen_epochs_ext"]

def save_checkpoint():
    with open(checkpoint_path, "wb") as f:
        pickle.dump({"extended_config": extended_config, "all_predictions_ext": all_predictions_ext,
                     "chosen_epochs_ext": chosen_epochs_ext}, f)

for name in EXTENDED_TRAINING_VARIANTS:
    if name in extended_config:
        print(f"{name}: search already done (checkpoint), skipping.")
        continue
    print(f"\n=== Validation search: {name} ===")
    t0 = _time.time()
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        _, best = search_patience_and_ceiling(
            ctor_for(name), internal_fit, internal_val, adj_physical, adj_corr, device,
            candidates=CANDIDATES, head_type=head_type_for[name], lr=best_lr[name],
            batch_size=BATCH_SIZE, seed=0,
        )
    extended_config[name] = best
    p, m, c = best["patience"], best["min_epochs_before_stopping"], best["ceiling"]
    print(f"  winner: patience={p}, min_epochs_before_stopping={m}, ceiling={c}  ({_time.time()-t0:.0f}s)")
    save_checkpoint()

for name in EXTENDED_TRAINING_VARIANTS:
    ext_name = f"{name} (extended)"
    if ext_name in all_predictions_ext:
        print(f"{ext_name}: refit already done (checkpoint), skipping.")
        continue
    cfg = extended_config[name]
    print(f"\n=== Training {ext_name} ===")
    all_predictions_ext[ext_name] = []
    chosen_epochs_ext[ext_name] = []
    head_type_for[ext_name] = head_type_for[name]
    for seed in range(EXT_N_SEEDS):
        t0 = _time.time()
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            _, best_epoch = train_one_run(
                ctor_for(name), internal_fit, internal_val, adj_physical, adj_corr, device,
                head_type=head_type_for[name], epochs=cfg["ceiling"], batch_size=BATCH_SIZE,
                lr=best_lr[name], patience=cfg["patience"],
                min_epochs_before_stopping=cfg["min_epochs_before_stopping"], seed=seed,
            )
            chosen_epochs_ext[ext_name].append(best_epoch)

            model = train_fixed_epochs(
                ctor_for(name), official_train, adj_physical, adj_corr, device,
                head_type=head_type_for[name], epochs=max(best_epoch, 1), batch_size=BATCH_SIZE,
                lr=best_lr[name], seed=seed,
            )

            test_loader = DataLoader(WindowDataset(*official_test), batch_size=BATCH_SIZE, shuffle=False)
            preds, trues = predict_all(model, test_loader, adj_physical.to(device), adj_corr.to(device),
                                        device, head_type_for[name])
        all_predictions_ext[ext_name].append((preds, trues))
        print(f"  seed {seed}: epoch_budget={best_epoch}  ({_time.time()-t0:.0f}s)")
        save_checkpoint()
    print(f"  epoch budgets across seeds: {chosen_epochs_ext[ext_name]}")

all_predictions.update(all_predictions_ext)
chosen_epochs.update(chosen_epochs_ext)
print("\nSection A (extended training) complete.")

## Section B — Ablations (Table VI)

In [ ]:
%%time

name = "CA-HGTN (full, beta-NLL isolated)"
print(f"\n=== Training {name} ===")
all_predictions[name] = []
chosen_epochs[name] = []
head_type_for[name] = "nb"
ctor = lambda: CAHGTN(num_stations=num_stations, hidden_dim=64, horizon=horizon,
                      head_type="nb", graph_mode="full", use_seasonal_residual=True,
                      output_bias_init=bias_init_official)
for seed in range(N_SEEDS):
    _, best_epoch = train_one_run(
        ctor, internal_fit, internal_val, adj_physical, adj_corr, device,
        head_type="nb", epochs=EPOCHS, batch_size=BATCH_SIZE,
        lr=best_lr["CA-HGTN (full)"], patience=PATIENCE, seed=seed,
        nb_loss_beta=1.0, nb_loss_aux_weight=0.0,
    )
    chosen_epochs[name].append(best_epoch)
    model = train_fixed_epochs(
        ctor, official_train, adj_physical, adj_corr, device,
        head_type="nb", epochs=max(best_epoch, 1), batch_size=BATCH_SIZE,
        lr=best_lr["CA-HGTN (full)"], seed=seed,
        nb_loss_beta=1.0, nb_loss_aux_weight=0.0,
    )
    test_loader = DataLoader(WindowDataset(*official_test), batch_size=BATCH_SIZE, shuffle=False)
    preds, trues = predict_all(model, test_loader, adj_physical.to(device), adj_corr.to(device), device, "nb")
    all_predictions[name].append((preds, trues))
print(f"  epoch budgets: {chosen_epochs[name]}")

In [ ]:
%%time

if not HAVE_PRECOMPUTED_GRAPHS:
    print(f"Skipping precomputed-graph ablation -- {CITY}_windows.npz lacks these fields. "
          f"Re-run notebook 00 for {CITY} to add them.")
else:
    for label, adj_second_graph in [("precomputed-correlation", adj_correlation_precomputed),
                                     ("precomputed-similarity", adj_similarity_precomputed)]:
        name = f"CA-HGTN (full, {label})"
        print(f"\n=== Training {name} ===")
        all_predictions[name] = []
        chosen_epochs[name] = []
        head_type_for[name] = "nb"
        ctor = lambda: CAHGTN(num_stations=num_stations, hidden_dim=64, horizon=horizon,
                              head_type="nb", graph_mode="full", use_seasonal_residual=True,
                              output_bias_init=bias_init_official)
        for seed in range(N_SEEDS):
            _, best_epoch = train_one_run(
                ctor, internal_fit, internal_val, adj_physical, adj_second_graph, device,
                head_type="nb", epochs=EPOCHS, batch_size=BATCH_SIZE,
                lr=best_lr["CA-HGTN (full)"], patience=PATIENCE, seed=seed,
            )
            chosen_epochs[name].append(best_epoch)
            model = train_fixed_epochs(
                ctor, official_train, adj_physical, adj_second_graph, device,
                head_type="nb", epochs=max(best_epoch, 1), batch_size=BATCH_SIZE,
                lr=best_lr["CA-HGTN (full)"], seed=seed,
            )
            test_loader = DataLoader(WindowDataset(*official_test), batch_size=BATCH_SIZE, shuffle=False)
            preds, trues = predict_all(model, test_loader, adj_physical.to(device), adj_second_graph.to(device),
                                        device, "nb")
            all_predictions[name].append((preds, trues))
        print(f"  epoch budgets: {chosen_epochs[name]}")

In [ ]:
%%time

temporal_variants = {
    "GCN+GRU (bidirectional-attention)": dict(bidirectional_attention=True),
    "GCN+GRU (stacked)": dict(temporal_mode="stacked_bidir_uni_attention"),
}
for name, extra_kwargs in temporal_variants.items():
    print(f"\n=== Training {name} ===")
    all_predictions[name] = []
    chosen_epochs[name] = []
    head_type_for[name] = "point"
    ctor = lambda: GCNGRUBaseline(num_stations=num_stations, hidden_dim=64, horizon=horizon,
                                  output_bias_init=bias_init_official, use_seasonal_residual=True,
                                  **extra_kwargs)
    for seed in range(N_SEEDS):
        _, best_epoch = train_one_run(
            ctor, internal_fit, internal_val, adj_physical, adj_corr, device,
            head_type="point", epochs=EPOCHS, batch_size=BATCH_SIZE,
            lr=best_lr["GCN+GRU baseline"], patience=PATIENCE, seed=seed,
        )
        chosen_epochs[name].append(best_epoch)
        model = train_fixed_epochs(
            ctor, official_train, adj_physical, adj_corr, device,
            head_type="point", epochs=max(best_epoch, 1), batch_size=BATCH_SIZE,
            lr=best_lr["GCN+GRU baseline"], seed=seed,
        )
        test_loader = DataLoader(WindowDataset(*official_test), batch_size=BATCH_SIZE, shuffle=False)
        preds, trues = predict_all(model, test_loader, adj_physical.to(device), adj_corr.to(device), device, "point")
        all_predictions[name].append((preds, trues))
    print(f"  epoch budgets: {chosen_epochs[name]}")
print("\nSection B (ablations) complete.")

## Section C — capacity scaling (exploratory, 1 seed)

In [ ]:
%%time

CAPACITY_LEVELS = [128, 256, 512]
capacity_results = {}
for hd in CAPACITY_LEVELS:
    name = f"GCN+GRU (hidden_dim={hd}, 1 seed, exploratory)"
    print(f"\n=== Training {name} ===")
    ctor = lambda hidden_dim=hd: GCNGRUBaseline(num_stations=num_stations, hidden_dim=hidden_dim, horizon=horizon,
                                                 output_bias_init=bias_init_official, use_seasonal_residual=True)
    _, best_epoch = train_one_run(
        ctor, internal_fit, internal_val, adj_physical, adj_corr, device,
        head_type="point", epochs=EPOCHS, batch_size=BATCH_SIZE,
        lr=best_lr["GCN+GRU baseline"], patience=PATIENCE, seed=0,
    )
    model = train_fixed_epochs(
        ctor, official_train, adj_physical, adj_corr, device,
        head_type="point", epochs=max(best_epoch, 1), batch_size=BATCH_SIZE,
        lr=best_lr["GCN+GRU baseline"], seed=0,
    )
    test_loader = DataLoader(WindowDataset(*official_test), batch_size=BATCH_SIZE, shuffle=False)
    preds, trues = predict_all(model, test_loader, adj_physical.to(device), adj_corr.to(device), device, "point")
    capacity_results[hd] = float(np.mean(np.abs(preds["pred"] - trues)))
    print(f"  hidden_dim={hd}: test MAE={capacity_results[hd]:.2f}, epoch budget={best_epoch}")
print("\nSection C (capacity scaling) complete:", capacity_results)

In [ ]:
%%time

import os, sys, json, time
import numpy as np, torch
from torch.utils.data import DataLoader

ROOT = "/Users/liam.yoon.28/Subway Prediction V2"   # <- folder containing cahgtn/ and Metro_Data/
CITY = "Hangzhou"          #"Shanghai", "Beijing"
N_SEEDS = 5

PATIENCE, MIN_EPOCHS, CEILING, LR = 15, 25, 100, 1e-3

if ROOT not in sys.path: sys.path.insert(0, ROOT)
from cahgtn.data_pipeline import (load_city, build_seasonal_profile, build_windows,
                                  _build_lagged_correlation_adjacency)
from cahgtn.model import CAHGTN
from cahgtn.train import WindowDataset, train_one_run, predict_all

device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

out_dir = os.path.join(ROOT, "_paper_revision_analysis"); os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f"graph_sensitivity_{CITY}.json")
results = json.load(open(out_path)) if os.path.exists(out_path) else {}

data = load_city(os.path.join(ROOT, "Metro_Data"), CITY)
ix, N = data.idx, data.num_stations

prof  = build_seasonal_profile(data, ix["internal_fit"])
fit_w = build_windows(data, ix["internal_fit"], prof)
val_w = build_windows(data, ix["internal_val"], prof)

A_train = data.adjacency_corr
A_ifit_np, _ = _build_lagged_correlation_adjacency(
    data.flow[ix["internal_fit"]], data.slot_id[ix["internal_fit"]], data.n_slots)
A_ifit = torch.tensor(np.asarray(A_ifit_np), dtype=torch.float32)
graphs = {"train": A_train, "ifit": A_ifit}

BIAS = tuple(float(v) for v in data.flow[ix["internal_fit"]].mean(axis=(0, 1)))
make_model = lambda: CAHGTN(num_stations=N, hidden_dim=64, horizon=4, head_type="nb",
                            graph_mode="full", use_seasonal_residual=True, output_bias_init=BIAS)

val_loader = DataLoader(WindowDataset(*val_w), batch_size=64, shuffle=False)

for seed in range(N_SEEDS):
    for g in ["train", "ifit"]:
        key = f"{g}|{seed}"
        if key in results: continue
        t0 = time.time(); A = graphs[g]
        model, best_epoch = train_one_run(
            make_model, fit_w, val_w, data.adjacency_physical, A, device, head_type="nb",
            epochs=CEILING, batch_size=32, lr=LR, patience=PATIENCE, seed=seed,
            min_epochs_before_stopping=MIN_EPOCHS)
        model.eval()
        p, y = predict_all(model, val_loader, data.adjacency_physical.to(device), A.to(device), device, "nb")
        results[key] = dict(graph=g, seed=seed, best_epoch=int(best_epoch),
                            val_mae_15min=float(np.abs(p["mu"][:, 0] - y[:, 0]).mean()),
                            val_mae_all=float(np.abs(p["mu"] - y).mean()))
        json.dump(results, open(out_path, "w"), indent=1)
        print(f"== {CITY} {key}: epoch={best_epoch} val MAE(15min)={results[key]['val_mae_15min']:.3f} [{time.time()-t0:.0f}s]")

# paired summary
rows = [(results[f"train|{s}"], results[f"ifit|{s}"]) for s in range(N_SEEDS)
        if f"train|{s}" in results and f"ifit|{s}" in results]
if rows:
    a = np.array([r[0]["val_mae_15min"] for r in rows]); b = np.array([r[1]["val_mae_15min"] for r in rows])
    d = b - a
    print(f"\nfull-train graph : {a.mean():.3f}   internal-fit graph: {b.mean():.3f}")
    print(f"paired diff (ifit - train): {d.mean():+.3f} ± {d.std(ddof=1) if len(d)>1 else float('nan'):.3f} (n={len(d)})")
    print("best epoch  train:", [r[0]['best_epoch'] for r in rows], " ifit:", [r[1]['best_epoch'] for r in rows])

## Save results

Everything accumulated above, for `02_Evaluate_and_Ablate.ipynb`.

In [ ]:
import pickle
with open(f"{CITY}_predictions.pkl", "wb") as f:
    pickle.dump({
        "all_predictions": all_predictions,
        "naive_test": naive_test,
        "head_type_for": head_type_for,
        "chosen_epochs": chosen_epochs,
        "epoch_times": epoch_times,
        "param_counts": param_counts,
        "capacity_results": capacity_results,
    }, f)
print(f"Saved {CITY}_predictions.pkl ({len(all_predictions)} variants) -- ")
print(f"proceed to 02_Evaluate_and_Ablate.ipynb, or come back here with the next CITY.")